## **Module 2: Cloud Composer**

### Overview of Google Cloud Composer 

Cloud Composer is a **fully managed workflow orchestration service** offered by Google Cloud. It's built on the popular open-source project **Apache Airflow**.

In essence, Cloud Composer allows you to:

1.  **Author workflows:** You define your data pipelines and complex sequences of tasks as Directed Acyclic Graphs (DAGs) using Python code. This gives you programmatic control over your entire workflow.
2.  **Schedule workflows:** You can set up your DAGs to run automatically at specific intervals (e.g., daily, hourly, or on specific days) or trigger them manually based on events.
3.  **Monitor workflows:** Cloud Composer provides a user-friendly web interface (the Airflow UI) where you can visualize your DAGs, track the status of individual tasks, view logs, and troubleshoot any issues.
4.  **Manage workflows:** It handles the underlying infrastructure for running Apache Airflow, so you don't have to worry about provisioning virtual machines, setting up databases, configuring schedulers, or managing upgrades.


![image.png](../artifacts/02_composer_architecture.png)


### Key Features and Benefits of Cloud Composer

* **Fully Managed:** This is its biggest selling point. Google handles the deployment, scaling, maintenance, and updates of the Airflow components (scheduler, web server, workers, database), freeing you from operational overhead.
* **Apache Airflow Compatibility:** Since it's based on Apache Airflow, you get all the benefits of the open-source project, including:
    * **Python-based DAGs:** Leverage your existing Python skills to define powerful and flexible workflows.
    * **Extensive Operators:** Airflow has a rich ecosystem of pre-built operators for interacting with various services, including a comprehensive set for Google Cloud services (BigQuery, Dataflow, Dataproc, Cloud Storage, Vertex AI, Pub/Sub, etc.), as well as external systems.
    * **Portability:** Your DAGs are standard Airflow DAGs, making them potentially portable to other Airflow environments if you ever need to migrate.
* **Deep Google Cloud Integration:** Cloud Composer offers seamless, out-of-the-box integration with other Google Cloud services. This makes it ideal for building end-to-end data pipelines, machine learning workflows, and IT automation tasks within the Google Cloud ecosystem.
* **Scalability:** Cloud Composer environments can automatically scale their worker nodes based on your workload, ensuring that your pipelines run efficiently even as demand changes.
* **Hybrid and Multi-Cloud Support:** While deeply integrated with Google Cloud, you can use Cloud Composer to orchestrate workflows that span across on-premises environments and even other cloud providers.
* **Robust Monitoring and Logging:** Integrates with Google Cloud's native logging (Cloud Logging) and monitoring (Cloud Monitoring) services, providing detailed insights into your workflow's performance and health.
* **Version Control for DAGs:** You store your DAG files in a Cloud Storage bucket, making it easy to manage versions and roll back changes.

**When to use Cloud Composer** Cloud Composer is particularly well-suited for:

* **Data Engineering Pipelines (ETL/ELT):** Automating the extraction, transformation, and loading of data from various sources into data warehouses like BigQuery.
* **Machine Learning Workflows (MLOps):** Orchestrating the entire ML lifecycle, from data preprocessing and feature engineering to model training, evaluation, deployment, and monitoring.
* **Batch Processing:** Scheduling and managing complex batch jobs.
* **Infrastructure Automation:** Automating provisioning, deprovisioning, and configuration of cloud resources.
* **Any complex, multi-step process:** Where you need reliable scheduling, dependency management, and clear visibility into execution status.

In essence, Cloud Composer acts as the brain for orchestrating complex, distributed tasks and processes, ensuring they run reliably, on schedule, and in the correct order across your Google Cloud (and potentially hybrid/multi-cloud) landscape.


### Cloud Composer Architecture: Managed Airflow Environment

At its core, Cloud Composer provisions and manages a complete Apache Airflow setup for you. This includes all the essential Airflow components, carefully orchestrated across Google Cloud services:

1.  **Google Kubernetes Engine (GKE) Cluster:**
    * **Foundation:** The entire Cloud Composer environment runs on a Google Kubernetes Engine cluster. This provides the scalable and robust container orchestration platform for Airflow's components.
    * **Airflow Components as Pods:** Airflow's core services—Schedulers, Workers, DAG Processors, and Triggers (in Composer 3 for deferrable operators)—run as pods within this GKE cluster.
    * **Managed Nodes:** Cloud Composer manages the GKE nodes (Compute Engine VMs) that host these pods, handling scaling, patching, and underlying infrastructure maintenance. You don't directly interact with the GKE cluster unless you need advanced debugging or custom configurations.

2.  **Airflow Scheduler(s):**
    * **Purpose:** The brain of Airflow. It continuously monitors the `dags/` folder in the Cloud Storage bucket (see below) for new or updated DAGs, parses them, creates DAG Runs, and queues up individual task instances for execution when their dependencies are met.
    * **Managed:** Cloud Composer ensures the scheduler is always running, highly available, and properly configured.

3.  **Airflow Workers:**
    * **Purpose:** These are the processes that actually execute your DAG's tasks. They pull tasks from a queue (typically a Redis queue, managed by Composer) and run the code defined in your operators.
    * **Scalability:** Cloud Composer can automatically scale the number of worker nodes (and thus worker processes) up and down based on your workload, ensuring efficient resource utilization and handling peak loads. This is a crucial managed feature for cost optimization and performance.

4.  **Airflow Web Server:**
    * **Purpose:** Provides the Airflow User Interface (UI), a web-based dashboard where you can visualize your DAGs, trigger runs, inspect task status, view logs, and manage connections/variables.
    * **Managed:** Cloud Composer hosts this web server (often on a Compute Engine instance or App Engine Flex in a Google-managed "tenant project") and handles its availability and security. Access is integrated with Google Cloud IAM.

5.  **Airflow Metadata Database (Cloud SQL):**
    * **Purpose:** This is the heart of Airflow's state management. It stores all information about DAGs, task runs, historical executions, connections, variables, and user data.
    * **Managed:** Cloud Composer provisions and manages a **Cloud SQL for PostgreSQL** instance for this database. Cloud SQL offers high availability, automatic backups, automated patching, and scalability, taking away the burden of database administration from you. This Cloud SQL instance typically resides in a Google-managed "tenant project" for enhanced security and unified access control.

6.  **Redis Queue (for Celery/Kubernetes Executors):**
    * **Purpose:** Acts as a message broker between the scheduler and the workers, holding tasks that are ready for execution.
    * **Managed:** Composer sets up and maintains this queue within the GKE cluster.

### Integration with Cloud Storage (Environment's Bucket)

Cloud Storage plays a fundamental role in how you interact with and operate your Cloud Composer environment:

* **DAG Storage:**
    * **Primary Interaction Point:** Each Cloud Composer environment is automatically associated with a dedicated Cloud Storage bucket (e.g., `gs://us-central1-your-composer-env-xxxxxx-bucket/`).
    * **`dags/` folder:** You upload your Airflow DAG `.py` files directly to the `dags/` subfolder within this bucket.
    * **Automatic Synchronization:** The Airflow scheduler continuously monitors this `dags/` folder. When you upload or update a DAG file, the scheduler automatically detects the change, parses the new DAG, and makes it available in the Airflow UI for scheduling and execution. This eliminates the need for manual deployment steps to the Airflow components.

* **Plugins Storage (`plugins/` folder):**
    * If you have custom Airflow plugins (e.g., custom operators, hooks, or sensors) that extend Airflow's functionality, you upload them to the `plugins/` folder in the same Cloud Storage bucket.
    * Cloud Composer automatically makes these plugins available to your Airflow environment.

* **Data Dependencies (`data/` folder):**
    * You can use other folders (like `data/`) within the bucket to store files that your DAGs might need during execution. For example, a CSV file for a data processing task, or a trained ML model artifact that needs to be loaded.
    * Tasks running in your Airflow workers can easily access these files using standard Google Cloud Storage client libraries.

* **Airflow Logs (`logs/` folder):**
    * While Cloud Composer also streams logs directly to Google Cloud Logging for real-time viewing, it also stores a copy of all Airflow task logs in the `logs/` folder within the Cloud Storage bucket. This provides a durable, historical record of all task executions.

* **Environment Snapshots:**
    * When you create a Cloud Composer environment snapshot, the snapshot data (including metadata database and DAGs) is stored in a separate location within your Cloud Storage bucket, enabling easy restoration.

### Customer Project vs. Tenant Project

Cloud Composer cleverly distributes its resources between two types of Google Cloud projects:

* **Customer Project (Your Project):**
    * This is the Google Cloud project where you create and manage your Cloud Composer environment.
    * It hosts components like the **GKE cluster nodes (Airflow workers)**, the **Cloud Storage bucket** for DAGs and logs, **Cloud Logging**, and **Cloud Monitoring**.
    * You have direct control and visibility over these resources, and they contribute to your project's billing.

* **Tenant Project (Google-Managed):**
    * This is a Google-managed project that's hidden from your direct view.
    * It hosts critical shared or sensitive components for your Composer environment, such as the **Cloud SQL metadata database**, **Cloud SQL storage for backups**, and often the **Airflow web server** (depending on the Composer version).
    * This separation provides enhanced security (isolating the database access) and simplifies management for Google. Costs for these tenant project resources are rolled into your overall Cloud Composer environment fee (DCU-based pricing in Composer 3).

### Summary of Managed Benefits

This architectural design means that as a Cloud Composer user, you largely interact with:

1.  **Your DAG files** in the Cloud Storage bucket.
2.  The **Airflow UI** (via the web server).
3.  **Google Cloud Console** for environment management, monitoring, and billing.

Google takes care of:

* Provisioning and maintaining the GKE cluster.
* Managing the Cloud SQL database.
* Ensuring high availability of Airflow components.
* Handling Airflow upgrades and patching.
* Scaling worker resources based on demand.
* Integrating logging and monitoring with Cloud Logging and Cloud Monitoring.

This managed approach significantly reduces the DevOps burden, allowing organizations to leverage the power of Apache Airflow for complex workflow orchestration without the complexities of self-hosting and maintaining it.


### Cloud Composer 2 vs. Cloud Composer 3: Architectural Comparison

When provisioning a new environment, it is important to understand the architectural improvements introduced in **Cloud Composer 3** compared to **Cloud Composer 2**:

| Feature / Dimension | Cloud Composer 2 | Cloud Composer 3 |
| :--- | :--- | :--- |
| **Underlying Compute** | Autopilot GKE cluster visible in your project | Fully abstracted infrastructure managed in Google tenant project |
| **VPC Networking Setup** | Requires VPC-native secondary IP ranges for GKE Pods & Services | Simplified networking via Private Service Connect / internal network attachment (no secondary IP ranges needed) |
| **Autoscaling** | Scales Airflow workers via GKE Autopilot | Fast, granular autoscaling of workers, schedulers, and triggerers |
| **DAG Processors** | Bundled with scheduler or separate component | Dedicated, independently scalable `dag-processor` component |
| **Security & Isolation** | Private IP requires GKE private cluster configuration | Private IP enabled effortlessly without managing GKE control plane CIDR blocks |
